# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/7ayder-99/flyrank_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
userdata.get('secretName')

SecretNotFoundError: Secret secretName does not exist.

In [3]:
%pip -q install duckdb

import duckdb
con = duckdb.connect()

# register your HF token from Colab Secrets (never paste it directly)
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# sanity check: near-free metadata queries first
con.sql("""
    SELECT COUNT(*) AS n, MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│    n    │   min_d    │   max_d    │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page, on one day, for one client — the grain of fact_content_daily_performance is report_date × client_id × content_id. Each row is a single day's observed activity for a single page belonging to a single client (impressions, clicks, sessions, engagement, and GSC position for that day).

Time window: I'm using the month=2026-03 partition — a mid-panel month, not the final month. The final month (June 2026, exposed separately as _sample) is deliberately excluded from this development work and treated as a sealed test month, since it's the natural outcome window for any past→future label I might build later.

I'll verify this claim right below with a grain-check query (GROUP BY client_id, content_id, report_date HAVING COUNT(*) > 1 should return zero rows) and a MIN/MAX(report_date) check to confirm the window is really March 2026.

In [5]:
con.sql(f"SELECT * FROM {REL} LIMIT 3").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

In [6]:
%pip -q install duckdb

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# Grain check
con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────┐
│ client_hash_id │ content_hash_id │ report_date │   c   │
│    varchar     │     varchar     │    date     │ int64 │
├────────────────┴─────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Draft answer — Section 2 (English, using the real column names):

Context (grouping/joining/splitting only — never features):

client_hash_id, content_hash_id — pseudonymous IDs, used for grouped client-holdout splits and joins to dim_content/dim_clients
report_date, month — used to define the time window, not fed to the model directly

Feature (knowable before the decision moment, safe to use):

gsc_impressions, gsc_clicks, gsc_avg_position — observed search performance for that day
ga4_sessions, ga4_engaged_sessions, ga4_pageviews — observed engagement for that day (only where ga4_data_available IS TRUE)
sessions_organic, sessions_ai (and the AI-source breakdown columns) — observed traffic-source mix
client_has_gsc, client_has_ga4 — availability flags, used to filter/segment, arguably context rather than a learned feature

Label / proxy (the thing I predict, or what it's computed from — never a feature):

Not present as a ready-made column in this table. I will aggregate gsc_impressions and gsc_clicks across day-ranges within the month to build the same trend-based priority proxy from w02/w03 — computed from the feature columns above, kept strictly separate from anything used as a feature.

Excluded (with a why):

gsc_sum_position — excluded because it's a raw sum, not a rate; using both gsc_sum_position and gsc_avg_position would double-encode the same signal (redundant, and sum is harder to interpret without impressions context)
scroll_events — excluded for this lane because it's missing whenever GA4 is unavailable, and adding it would introduce the same missingness-follows-availability problem the data dictionary warns about, without adding much signal beyond ga4_engaged_sessions
Any FlyRank product flag or health score (not in this table, but a standing exclusion) — these are outputs of an existing rule; using them as features would mean the model just re-learns a prior decision, not a new one

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
features_sql = f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(gsc_avg_position) AS avg_position_month,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_sessions_month,
        COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END) AS days_with_impressions
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id
"""

feat_df = con.sql(features_sql).df()
feat_df.head(10)




FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_month,clicks_month,avg_position_month,engaged_sessions_month,days_with_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,0.0,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,0.0,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,0.0,31
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,0.0,21
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,0.0,31
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,9.445635,0.0,31
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,1.0,6.014516,0.0,31
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,9.155335,0.0,31
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,5.258331,0.0,31


impressions_month — knowable because it's a sum of already-observed daily GSC impressions within the month, no future data involved.
clicks_month — same reasoning: aggregated from past daily clicks.
avg_position_month — knowable because it's the average of daily GSC positions already logged at the time of review.
engaged_sessions_month — knowable from already-recorded GA4 sessions, filtered to rows where ga4_data_available IS TRUE so I'm not treating unavailable data as zero.
days_with_impressions — knowable because it's a count over days that have already elapsed within the window.

In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# build the proxy label FIRST — using the relaxed "2 out of 3 signals" version
feat_df["ctr_month"] = np.where(feat_df["impressions_month"] > 0,
                                  feat_df["clicks_month"] / feat_df["impressions_month"], 0)

feat_df["priority_signals"] = (
    (feat_df["avg_position_month"] > 10).astype(int) +
    (feat_df["ctr_month"] < feat_df["ctr_month"].median()).astype(int) +
    (feat_df["impressions_month"] > feat_df["impressions_month"].median()).astype(int)
)

# priority if at least 2 out of 3 signals fire
feat_df["is_priority"] = (feat_df["priority_signals"] >= 2).astype(int)

print("is_priority distribution:")
print(feat_df["is_priority"].value_counts())

# --- HONEST baseline: only the 5 real features, no leakage ---
honest_cols = ["impressions_month", "clicks_month", "avg_position_month",
               "engaged_sessions_month", "days_with_impressions"]

X_honest = feat_df[honest_cols].fillna(0)
y = feat_df["is_priority"]

model = LogisticRegression(max_iter=1000).fit(X_honest, y)
honest_auc = roc_auc_score(y, model.predict_proba(X_honest)[:, 1])
print("\nHonest AUC:", honest_auc)

# --- THE TRAP: deliberately leak the label's own ingredient back in as a "feature" ---
feat_df["leaky_ctr_month"] = feat_df["ctr_month"]  # this is literally inside how is_priority was built

X_leaky = feat_df[honest_cols + ["leaky_ctr_month"]].fillna(0)
model_leaky = LogisticRegression(max_iter=1000).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, model_leaky.predict_proba(X_leaky)[:, 1])
print("Leaky AUC (watch it jump toward 1.0):", leaky_auc)

# --- delete the leak, keep the honest number ---
del feat_df["leaky_ctr_month"]
print("\nFinal reported metric (honest, kept):", honest_auc)
print("Kept features:", honest_cols)

is_priority distribution:
is_priority
0    257691
1     73746
Name: count, dtype: int64

Honest AUC: 0.9868361877487735
Leaky AUC (watch it jump toward 1.0): 0.986836134601174

Final reported metric (honest, kept): 0.9868361877487735
Kept features: ['impressions_month', 'clicks_month', 'avg_position_month', 'engaged_sessions_month', 'days_with_impressions']


In [11]:
# an even more direct leak: the pre-threshold signal count itself (almost literally the label)
feat_df["leaky_priority_signals"] = feat_df["priority_signals"]

X_super_leaky = feat_df[honest_cols + ["leaky_priority_signals"]].fillna(0)
model_super_leaky = LogisticRegression(max_iter=1000).fit(X_super_leaky, y)
super_leaky_auc = roc_auc_score(y, model_super_leaky.predict_proba(X_super_leaky)[:, 1])
print("Super-leaky AUC (near-perfect):", super_leaky_auc)

del feat_df["leaky_priority_signals"]

Super-leaky AUC (near-perfect): 0.9999999999999999


The leakage trap, performed: My "honest" baseline already scored AUC ≈ 0.987 — a red flag on its own, since a genuinely hard ranking problem shouldn't be nearly solved by 5 simple aggregate features. Investigating showed why: my proxy label (is_priority) was built directly from avg_position_month, ctr_month, and impressions_month — the same three signals present in my feature set. The model was reconstructing my own threshold rule, not learning anything about the world.

To make the trap explicit, I added priority_signals (the pre-threshold signal count — almost the label itself) as a deliberate leaky feature. AUC jumped to 0.9999999999999999 — essentially perfect, because the feature is the label in disguise.

I then removed leaky_priority_signals and kept only the 5 honest features. But the deeper lesson is that even the "honest" 0.987 number is not trustworthy on its own — a proxy label built from the same raw signals used as features will always look artificially strong. Before this proxy can be used for real prioritization, it needs to be redefined using a genuinely separate signal (e.g., an outcome measured in a later time window than the features), not a same-month threshold rule over the same columns.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation 1 (from the trap above): Any priority proxy built as a same-window threshold rule over aggregate GSC/GA4 signals will trivially "predict" itself — this slice of data cannot validate a same-month label. A trustworthy proxy needs a genuinely future outcome window, which month=2026-03 alone cannot provide.

Limitation 2: ga4_data_available was NULL (not false) for a meaningful share of rows in this month, meaning client-level GA4 history doesn't start uniformly — clients with client_has_ga4 = false contribute zero real engagement signal for the whole month, which could bias engaged_sessions_month toward under-representing genuinely low-GA4-coverage clients rather than genuinely low engagement.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.